# Tunix — Tesla T4 benchmark

Select **Runtime → Change runtime type → T4 GPU**. Accept the Llama 3.2 1B Instruct license on Hugging Face before running. Run cells in order. Benchmarking needs only your Hugging Face token; it does not call Meta. Fresh runs use three repeats and separate result sessions. Raw speedups are not quality approval.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path('/content/Tunix')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/karim-elmaraghy/Tunix.git', str(repo)], check=True)
# Existing checkouts are preserved; use a fresh runtime for the latest main.
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[gpu]'], check=True)


In [ ]:
subprocess.run(['tunix', 'doctor'], check=True)
if not Path('tunix.json').exists():
    subprocess.run(['tunix', 'init'], check=True)
import getpass
os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face read token: ')


## Measure and export

A successful benchmark is measured evidence, not an approved winner. Failed profiles and blocked gates must remain visible. Export files use unique names to preserve prior runs.


In [ ]:
run = subprocess.run(['tunix', 'benchmark'], check=False)
print('Benchmark exit:', run.returncode)


In [ ]:
import uuid
from google.colab import files
zip_path = '/content/tunix-results-' + uuid.uuid4().hex[:12] + '.zip'
subprocess.run(['tunix', 'export', '--output', zip_path], check=True)
files.download(zip_path)


## Optional: Meta-assisted optimization

Run these cells only if you intend to call the Meta API. They may use API credits. Calibration must pass before candidate search; a blocked result is not success. No configuration is deployed. Export again afterward to download the optimization session.


In [ ]:
RUN_OPTIMIZATION = False
if RUN_OPTIMIZATION:
    os.environ['MODEL_API_KEY'] = getpass.getpass('Meta API key: ')
    try:
        run = subprocess.run(['tunix', 'optimize', '--budget', '8'], check=False)
        print('Optimization exit:', run.returncode)
    finally:
        os.environ.pop('MODEL_API_KEY', None)


## View the downloaded results locally

In your Tunix checkout, run `python -m pip install -e .`, then `tunix init` once and `tunix report`. Open the displayed local URL, select **Open Workbench**, and import the downloaded ZIP. To view bundled measurements without a GPU, use `tunix report --replay`.
